In [ ]:
import time
import cv2
import numpy as np
from jetbot import Camera

class ObstacleDetector:
    def __init__(self, width=224, height=224):
        self.camera = Camera.instance(width=width, height=height)

        def get_roi(image):
            h, w, _ = image.shape
            y1 = int(h * 0.65)
            y2 = int(h * 0.92)
            x1 = int(w * 0.45)
            x2 = int(w * 0.55)
            return image[y1:y2, x1:x2]

        self.get_roi = get_roi

        print("Kalibrerer gulv... sørg for at der IKKE står noget foran robotten.")
        time.sleep(2)

        low_frames = []
        for _ in range(25):
            frame = self.camera.value
            roi = self.get_roi(frame)
            gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY).astype(np.float32)
            low_frames.append(gray)
            time.sleep(0.05)

        self.bg_gray = np.mean(low_frames, axis=0)
        print("Kalibrering færdig.")

        # parametre
        self.PIXEL_DIFF_PIXEL_THRESHOLD = 10.0
        self.FRACTION_CLOSE_THRESHOLD   = 0.40
        self.MIN_FRAC_FOR_DISTANCE      = 0.05
        self.K_FRAC                     = 6.0

    def detect(self):
        """Returnerer: obstacle_close, frac, distance_cm, frame"""
        frame = self.camera.value
        roi = self.get_roi(frame)
        gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY).astype(np.float32)

        diff_pixels = np.abs(gray - self.bg_gray)
        mask = diff_pixels > self.PIXEL_DIFF_PIXEL_THRESHOLD

        frac = float(np.mean(mask))
        obstacle_close = frac > self.FRACTION_CLOSE_THRESHOLD

        distance_cm = None
        if frac > self.MIN_FRAC_FOR_DISTANCE:
            distance_cm = self.K_FRAC / frac

        return obstacle_close, frac, distance_cm, frame

    def stop(self):
        self.camera.stop()
